<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: The precision–recall trade-off

**[The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/)** · Machine Learning: From Problem to Reliable Model · Module 3, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** move a threshold to trade recall for precision, measure the top of a ranked list with precision@k and recall@k, and summarise every threshold with the precision–recall curve, average precision, the ROC curve and ROC AUC. The notebook ends with the module practice.

| | |
|---|---|
| **Time** | About 70 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The confusion matrix, precision, recall and Larkfield's points score, from [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/). The 20% capacity, from [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). |
| **You do not need** | The local project. The setup below downloads the data. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M03-L04-precision-recall-trade-off/precision-recall-trade-off-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Move the threshold on ten tickets

A score ranks the tickets; a **threshold** turns it into flags: flag every ticket with a score at or above it. For Larkfield's points score (0 to 4 points), the threshold is a whole number of points, the cut-off.

The next cell defines the points score and the ten tickets of [Classification metrics](https://learning.nextia-ai.com/courses/ml/m03/classification-metrics/), then prints precision and recall at every cut-off, from the strictest to the widest.

> **Predict.** As the cut-off goes down from 4 to 0, does recall go up or down? Precision? Then run the cell.

In [ ]:
TARGET = "escalated_72h"

def points_score(tickets):
    """Larkfield's points score: 0 to 4 points per ticket."""
    points = 2 * (tickets["priority"] == 1)
    points += tickets["team"].isin(["payment", "warranty"])
    points += tickets["prior_escalations_90d"] >= 1
    return points.astype(int).rename("points")

tickets_10 = pd.DataFrame({
    "priority": [1, 1, 1, 1, 3, 2, 3, 2, 3, 3],
    "team": ["payment", "delivery", "warranty", "returns", "payment", "account", "delivery", "warranty", "returns", "account"],
    "prior_escalations_90d": [1, 0, 0, 1, 0, 0, 2, 1, 0, 1],
    "escalated_72h": [1, 0, 1, 0, 1, 0, 0, 1, 0, 0],
}, index=[f"T-{i}" for i in range(1, 11)])
points_10 = points_score(tickets_10)
y_10 = tickets_10[TARGET]
for cut_off in range(4, -1, -1):
    flags = points_10 >= cut_off
    tp = int((flags & (y_10 == 1)).sum())
    print(f"{cut_off}+ points: flagged {flags.sum():2d}, precision {tp / flags.sum():.2f}, recall {tp / y_10.sum():.2f}")

> **Check.** You should see recall go up from 0.25 (4+ points) to 1.00 (1+), and precision go down from 1.00 to 0.40 (all tickets). T-2 and T-8 both have 2 points, so no cut-off flags one without the other. A points score has few values, so tickets **tie**.

This is the **precision–recall trade-off**: a wider net catches more fish and more junk. Only a better score improves both.

Two numbers summarise every threshold at once:

- **Average precision (AP)**: go down the list; each time you catch one more escalation, note the precision; take the mean, each precision weighted by how much recall grew. Here: (1.00 + 0.67 + 0.60 + 0.50) ÷ 4.
- **ROC AUC**: the chance that a random escalated ticket gets a higher score than a random other ticket (ties count half). Here: 19 of 24 pairs.

Run the cell to get both from scikit-learn.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

print("AP", round(average_precision_score(y_10, points_10), 3), "| ROC AUC", round(roc_auc_score(y_10, points_10), 3))

> **Check.** You should see `AP 0.692 | ROC AUC 0.792`.

## 2. A ranked list: precision@k and recall@k

A search engine ranks pages. For a list, the cut is a number of results, **k**:

- **precision@k**: of the top k results, the share that are relevant;
- **recall@k**: of all relevant results that exist, the share that are in the top k.

The RAG course uses them to judge retrieval: is the passage with the answer among the k passages given to the language model?

Here is one search: 10 results, where 1 means relevant. A fifth relevant page exists, below rank 10.

> **Your turn.** Compute precision@3 and recall@3 of `relevant`. Put them in `precision_at_3` and `recall_at_3`, each rounded to 3 decimals. Then run the check cell.

In [ ]:
relevant = np.array([1, 1, 0, 1, 0, 0, 1, 0, 0, 0])
all_relevant = 5
precision_at_3 = round(relevant[:3].sum() / 3, 3)
recall_at_3 = round(relevant[:3].sum() / all_relevant, 3)
print(precision_at_3, recall_at_3)

In [ ]:
expect_hash('(precision_at_3, recall_at_3)', (precision_at_3, recall_at_3), '84daca0907961c20')

> **Predict.** For k = 10, will precision@k be higher or lower than at k = 3? And recall@k? Then run the cell.

In [ ]:
for k in (1, 3, 5, 10):
    print(f"k = {k:2d}: precision@k {relevant[:k].sum() / k:.2f}, recall@k {relevant[:k].sum() / all_relevant:.2f}")

> **Check.** You should see precision@k fall from 1.00 to 0.40, and recall@k rise from 0.20 to 0.80. A longer list never loses a relevant page.

## 3. The trade-off on May

Now the 1,179 May tickets of the validation set. The next cell loads the data (with `prior_escalations_90d` from `extra_features.csv`), defines `flag_report()` from the last lesson, and prints every cut-off.

> **Predict.** The senior team can review at most 20% of the tickets. Which cut-off is the lowest one that fits? Then run the cell.

In [ ]:
def load(part):
    """One part of the split, with prior_escalations_90d from extra_features.csv."""
    tickets = pd.read_csv(f"data/{part}.csv", keep_default_na=False, na_values=[""])
    extra = pd.read_csv("data/extra_features.csv", usecols=["ticket_id", "prior_escalations_90d"])
    return tickets.merge(extra, on="ticket_id", how="left")

def flag_report(y, flags):
    """Compare 0/1 flags with the real 0/1 answers: four counts and five measures."""
    y, flags = np.asarray(y, dtype=bool), np.asarray(flags, dtype=bool)
    caught, false_alarms = int((flags & y).sum()), int((flags & ~y).sum())
    missed, correctly_left = int((~flags & y).sum()), int((~flags & ~y).sum())
    precision = caught / (caught + false_alarms) if caught + false_alarms else None
    recall = caught / (caught + missed)
    f1 = 2 * precision * recall / (precision + recall) if caught else 0.0
    return {"flagged": caught + false_alarms, "share flagged": round((caught + false_alarms) / len(y), 3),
            "caught": caught, "false alarms": false_alarms, "missed": missed, "correctly left": correctly_left,
            "precision": round(precision, 3) if precision is not None else None, "recall": round(recall, 3),
            "F1": round(f1, 3), "accuracy": round((caught + correctly_left) / len(y), 3)}

valid = load("valid")
y = valid[TARGET]
points = points_score(valid)
for cut_off in range(4, -1, -1):
    r = flag_report(y, points >= cut_off)
    print(f"{cut_off}+ points: flagged {r['flagged']:4d} ({r['share flagged']:.1%})  caught {r['caught']:3d}  false alarms {r['false alarms']:3d}"
          f"  precision {r['precision']:.3f}  recall {r['recall']:.3f}")

> **Check.** You should see five rows. 3+ points flags 116 tickets (9.8%), with precision 0.457 and recall 0.290. 2+ points flags 327 (27.7%), over the capacity.

So the best cut-off for the capacity is 3 points. The next step flags 27.7%, so 119 of the 235 possible reviews stay unused: 211 tickets tie at 2 points. A **tie rule** fixes this: flag the 3+ tickets, then the 2-point tickets **in arrival order** until the capacity is full. Section 4 computes it. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) has an explorer with these numbers: move the cut-off there.

## 4. The curves and their summaries

`precision_recall_curve()` returns the precision and recall at every threshold. Run the cell.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y, points)
for p, r, t in zip(precision, recall, thresholds):
    print(f"{t} points or more: precision {p:.3f}, recall {r:.3f}")

> **Check.** You should see the same five cut-offs, from `0 points or more: precision 0.155, recall 1.000` to `4 points or more: precision 0.562, recall 0.098`.

The next cell draws the **precision–recall curve** (recall across, precision up) and the **ROC curve** (false-positive rate across, recall up), with the priority-1 rule as a single point. Run it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

rule = flag_report(y, valid["priority"] == 1)
false_positive_rate, true_positive_rate, _ = roc_curve(y, points)
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
left.plot(recall, precision, "o-", label="points score")
left.axhline(y.mean(), color="grey", linestyle=":", label="random flags")
left.plot(rule["recall"], rule["precision"], "ks", label="priority-1 rule")
left.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve")
right.plot(false_positive_rate, true_positive_rate, "o-", label="points score")
right.plot([0, 1], [0, 1], color="grey", linestyle=":", label="guessing")
right.plot(rule["false alarms"] / (rule["false alarms"] + rule["correctly left"]), rule["recall"], "ks", label="priority-1 rule")
right.set(xlabel="False-positive rate", ylabel="True positive rate (recall)", title="ROC curve")
left.legend()
right.legend()
plt.show()

> **Check.** On the left, the curve falls from the top left to the bottom right, and the rule's square is below it. On the right, the curve bends up above the dotted diagonal, and the rule's square is just below the curve.

Now the two summaries, next to their floors.

> **Predict.** A score that cannot rank has ROC AUC 0.5 and AP equal to the share of escalated tickets. Is the points score's AP closer to 0.155 or to 1? Then run the cell.

In [ ]:
print("average precision:", round(average_precision_score(y, points), 3), "| floor:", round(y.mean(), 3))
print("ROC AUC:", round(roc_auc_score(y, points), 3), "| floor: 0.5")

> **Check.** You should see `average precision: 0.31 | floor: 0.155` and `ROC AUC: 0.715 | floor: 0.5`.

Last, precision@k and recall@k for the top of the ranked May tickets. `at_k()` sorts by points and keeps the arrival order among tickets with the same points: the tie rule. 32 tickets have 4 points and 116 have 3 or more; k = 235 is the full capacity with the tie rule. Run the cell.

In [ ]:
def at_k(y, scores, k):
    """Precision and recall of the k highest scores."""
    top = np.argsort(-np.asarray(scores), kind="stable")[:k]
    caught = int(np.asarray(y)[top].sum())
    return round(caught / k, 3), round(caught / int(np.sum(y)), 3)

for k in (32, 116, 235):
    print(f"k = {k}: precision@k, recall@k =", at_k(y, points, k))

> **Check.** You should see `(0.562, 0.098)`, `(0.457, 0.29)` and `(0.357, 0.459)`. The first two are the cut-offs 4 and 3: a k and a threshold are two ways to cut the same list. With the tie rule, the full capacity catches 46% of the escalations, more than the target of 40%.

## 5. Change one thing: rarer escalations

Keep all 996 May tickets that did not escalate, but only 61 of the 183 escalations, picked at random. The points do not change.

> **Predict.** Does ROC AUC go up, down or stay about the same? And AP? Then run the cell.

In [ ]:
rng = np.random.default_rng(0)
escalated = np.where(y == 1)[0]
others = np.where(y == 0)[0]
keep = np.concatenate([rng.choice(escalated, 61, replace=False), others])
y_rare, points_rare = y.to_numpy()[keep], points.to_numpy()[keep]
print(f"escalated: {y_rare.mean():.1%}")
print("ROC AUC:", round(roc_auc_score(y_rare, points_rare), 3))
print("average precision:", round(average_precision_score(y_rare, points_rare), 3))

> **Check.** You should see `escalated: 5.8%`, `ROC AUC: 0.704` and `average precision: 0.135`.

ROC AUC compares pairs, so it hardly changes. AP falls with the rate, because each flag finds an escalation less often; its floor falls to 0.058 too. With rare positives, a good ROC AUC can hide a poor precision at the top. Compare AP only on the same tickets, next to its floor.

## 6. Failure case: precision at different workloads

Tomás says: "At 3 points or more, the points score has precision 0.457; the rule has 0.374. The points score is 22% more precise."

> **Predict.** How many escalations does each one catch? Then run the cell.

In [ ]:
for name, flags in [("rule: priority 1", valid["priority"] == 1), ("points: 3 or more", points >= 3)]:
    report = flag_report(y, flags)
    print(f"{name:18s} flagged {report['flagged']:3d} | precision {report['precision']} | recall {report['recall']}")
print("points, tie rule, 163 flags: precision, recall =", at_k(y, points, 163))

> **Check.** You should see `flagged 163 | precision 0.374 | recall 0.333` for the rule, `flagged 116 | precision 0.457 | recall 0.29` for the points score, and `(0.405, 0.361)` for the points score with the rule's 163 flags.

A stricter cut-off almost always raises precision. Tomás compares two different workloads: 116 flags against 163. **Compare at the same workload**: with 163 flags and the tie rule, the points score beats the rule on both numbers. Or compare across every threshold (the curve, or AP).

## 7. Module practice: two flag rules for the capacity

> **Your turn.** Find the lowest cut-off of the points score that flags at most 20% of the May tickets. Hint: `(points >= c).mean()` is the share flagged at cut-off `c`; try `c` from 0 to 5. Put the cut-off in `cut_20`, as an `int`. Then run the check cell.

In [ ]:
cut_20 = min(c for c in range(6) if (points >= c).mean() <= 0.20)
print("cut-off:", cut_20)

In [ ]:
expect_hash('cut_20', cut_20, '4e07408562bedb8b')

Now compare the ways to flag at the capacity: the rule, the cut-off alone, and the cut-off with the tie rule. Run the cell. Then write a short note for Grace in the empty Markdown cell below it: for each rule, the share flagged, precision and recall, and whether it meets the target (at most 20% flagged, recall at least 0.40).

In [ ]:
for name, flags in [("rule: priority 1", valid["priority"] == 1), ("points: 3 or more", points >= 3)]:
    report = flag_report(y, flags)
    print(f"{name:18s} {report['share flagged']:.1%} flagged | precision {report['precision']} | recall {report['recall']}")
print("tie rule, 235 flags: precision, recall =", at_k(y, points, 235))

> **Check.** You should see `13.8% flagged | precision 0.374 | recall 0.333` for the rule, `9.8% flagged | precision 0.457 | recall 0.29` for the points score, and `(0.357, 0.459)` with the tie rule.

*Write your note for Grace here (double-click to edit).*

The model note is on the lesson page, in **Module practice**. In short: the rule and the cut-off alone both stay below the 40% recall target. With the tie rule, the points score uses the whole capacity and catches 46% of the escalations on May, which meets the target. Module 4 builds models with finer scores, which need no tie rule.

## Next

You can now move a threshold for a decision, measure a ranked list with precision@k and recall@k, and compare scores with average precision and ROC AUC. That ends Module 3. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) has the second module task and the module check, which count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Regression intuition](https://learning.nextia-ai.com/courses/ml/m04/regression-intuition/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.